# OCR + LLM Document Pipeline -- Colab walkthrough

This is the original exploratory Colab notebook this project was built from. For a reusable, parameterized, command-line version of the same pipeline, see [`src/ocr_llm_pipeline.py`](../src/ocr_llm_pipeline.py) -- it exposes the same steps as `--input-dir`/`--markdown-dir`/`--reports-dir` CLI arguments instead of the hardcoded Google Drive paths below.

Paths in this notebook default to the original Colab layout (`/content/drive/MyDrive/...`) but can be overridden via environment variables so the notebook also runs outside Colab, given a mounted or local directory in the same shape.

In [ ]:
!pip install docling rapidocr-onnxruntime opencv-python-headless


In [ ]:
!pip install llama-index llama-index-llms-ollama pandas

In [ ]:
!pip install llama-index-llms-openai-like

In [ ]:
import os
from pathlib import Path
import cv2
import tempfile
import time
import json
from typing import List
import pandas as pd
from google.colab.patches import cv2_imshow

from llama_index.llms.ollama import Ollama
import socket
from llama_index.llms.openai_like import OpenAILike
from google.colab import userdata

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, ImageFormatOption
from docling_core.types.doc import ImageRefMode

In [ ]:
def preprocess_image(image_path):

    im = cv2.imread(image_path)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

    # Sharpen
    sharpen = cv2.GaussianBlur(gray, (0, 0), 3)
    sharpen = cv2.addWeighted(gray, 1.5, sharpen, -0.5, 0)

    # Apply adaptive threshold filtering
    thresh = cv2.adaptiveThreshold(sharpen, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 15)

    # Create a temporary file to hold the preprocessed image
    temp_dir = tempfile.gettempdir()
    preprocessed_path = os.path.join(temp_dir, f"preprocessed_{Path(image_path).name}")

    cv2.imwrite(preprocessed_path, thresh)

    return preprocessed_path

In [ ]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True

pipeline_options.do_formula_enrichment = True       # Formula recognition
pipeline_options.generate_picture_images = True     # Extract embedded images

# Configure RapidOCR (PaddleOCR)
ocr_options = RapidOcrOptions()
pipeline_options.ocr_options = ocr_options

# Apply the configuration to the supported formats
format_options = {
    InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    InputFormat.IMAGE: ImageFormatOption(pipeline_options=pipeline_options)
}

converter = DocumentConverter(format_options=format_options)

In [ ]:
def run_ocr_pipeline(file_path, converter, output_dir, Assets = False):
    """
    Takes a file, preprocesses it if needed, and runs it through Docling. If
    Assets=True, saves images to a separate folder and returns text with image
    references instead of embedded images.
    """
    path = Path(file_path)
    suffix = path.suffix.lower()
    is_image = suffix in [".png", ".jpg", ".jpeg"]
    target_path = str(path)
    temp_file_to_clean = None

    if is_image:
        target_path = preprocess_image(str(path))
        temp_file_to_clean = target_path

    result = converter.convert(target_path)

    if temp_file_to_clean and os.path.exists(temp_file_to_clean):
        os.remove(temp_file_to_clean)

    doc_dict = result.document.export_to_dict()

    if Assets == True:
        # Create an image folder for the current document
        assets_dir = output_dir / f"{path.stem}_assets"
        assets_dir.mkdir(parents=True, exist_ok=True)
        md_path = output_dir / f"{path.stem}.md"

        # Save Markdown with external image references
        result.document.save_as_markdown(
            filename=md_path,
            image_mode=ImageRefMode.REFERENCED,
            artifacts_dir=assets_dir
        )

        # Read the file back so we can print it to the console
        with open(md_path, "r", encoding="utf-8") as f:
            markdown_text = f.read()
    else:
        # If Assets=False, images are embedded as base64
        markdown_text = result.document.export_to_markdown(image_mode=ImageRefMode.EMBEDDED)

    return markdown_text, doc_dict

In [ ]:
# Override any of these via environment variables to run outside Colab.
input_folder = Path(os.environ.get("OCR_INPUT_DIR", "/content/drive/MyDrive/ocr_samples"))
allowed_extensions = {'.pdf', '.jpg', '.jpeg', '.png'}

output_folder = Path(os.environ.get("OCR_MARKDOWN_DIR", "/content/drive/MyDrive/markdown_results"))
output_folder.mkdir(parents=True, exist_ok=True)

files_to_process = [f for f in input_folder.iterdir() if f.is_file() and f.suffix.lower() in allowed_extensions]
print(f"Files found to process: {len(files_to_process)}")

for file_path in files_to_process:
    print(f"\nProcessing file: {file_path.name}")
    print("-" * 60)
    start_time = time.time()

    # Assets=True saves the Markdown file and images to a separate folder
    md_result, doc_dict = run_ocr_pipeline(str(file_path), converter, output_dir=output_folder, Assets=True)

    processing_time = time.time() - start_time
    minutes = int(processing_time // 60)
    seconds  = processing_time % 60

    json_output_path = output_folder / (file_path.stem + ".json")

    # Save the JSON file
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(doc_dict, f, ensure_ascii=False, indent=2)

    print(f"Processing time: {minutes} min {seconds:.2f} sec")
    print(f"Images saved to: {output_folder / f'{file_path.stem}_assets'}")
    print("-" * 60)

In [ ]:
OLLAMA_MODEL = "gemma4:31b-cloud"  # "gpt-oss:20b-cloud"

In [ ]:
def run_llama_extraction(markdown_text, model_name = "gemma4:31b-cloud"):
    """
    Query the Ollama Cloud API.
    """
    start_time = time.time()
    print(f"[LlamaIndex] Connecting to Ollama Cloud for model {model_name}...")

    # Fetch the API key
    api_key = userdata.get('ocr_olama')
    if not api_key:
        raise ValueError("Add the 'ocr_olama' secret with your Ollama API key.")

    llm = OpenAILike(
        model=model_name,
        api_key=api_key,
        api_base="https://ollama.com/v1",  # Official Ollama Cloud endpoint
        is_chat_model=True,
        temperature=0.1,
        max_tokens=2048,
        timeout=360.0
    )

    # The prompt below is deliberately written in Russian and asks for a
    # Russian-language report -- this pipeline targets Russian-language
    # document analysis, so the output language is a feature, not
    # untranslated text.
    prompt = f"""
    Перед тобой структурированное представление документа в формате Markdown.
    Твоя задача — внимательно проанализировать его и составить краткий аналитический отчет/резюме на русском языке (1-2 абзаца).

    Сфокусируйся на:
    - Основной цели документа / исследования.
    - Ключевых выводах, результатах или метриках.

    Важные требования:
    - Пиши только связный текст отчета на русском языке.
    - Не добавляй вводных фраз ("Вот ваш отчет:", "Конечно, я помогу" и т.д.).

    Документ для анализа:
    ---
    {markdown_text}
    ---
    """

    print("[LlamaIndex] Sending request to Ollama...")
    response = llm.complete(prompt)
    report = response.text.strip()

    end_time = time.time()
    processing_time = end_time - start_time
    print(f"[LlamaIndex] Analysis completed in {processing_time:.2f} sec")

    return report

In [ ]:
input_md_folder = Path(os.environ.get("OCR_MARKDOWN_DIR", "/content/drive/MyDrive/markdown_results"))
reports_folder = Path(os.environ.get("OCR_REPORTS_DIR", "/content/drive/MyDrive/final_results"))
reports_folder.mkdir(parents=True, exist_ok=True)


md_files = list(input_md_folder.glob("*.md"))
print(f"\nMarkdown files found to analyze: {len(md_files)}")

for md_file_path in md_files:
    print(f"\nAnalyzing file: {md_file_path.name}")
    print("-" * 60)

    try:
        with open(md_file_path, "r", encoding="utf-8") as f:
            markdown_content = f.read()

        # Generate a report (uses the model from OLLAMA_MODEL)
        extracted_report = run_llama_extraction(markdown_content, model_name=OLLAMA_MODEL)

        # Save the text report
        txt_output_path = reports_folder / (md_file_path.stem + "_report.txt")
        with open(txt_output_path, "w", encoding="utf-8") as f:
            f.write(extracted_report)

        print(f"Text report saved: {txt_output_path}")
        print("\n=== SHORT ANALYTICAL REPORT ===")
        print(extracted_report)
        print("-" * 60)

    except Exception as e:
        print(f"[ERROR] Could not process file {md_file_path.name}: {e}")
        print("Moving on to the next file...")
        print("-" * 60)
        continue

print("\nProcessing and report generation complete.")